## Loading the dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import mailbox

mboxfile = "/Users/grindhardt/Desktop/GMAIL DATA/All mail Including Spam and Trash-002.mbox"
mbox = mailbox.mbox(mboxfile)
mbox

In [3]:
for key in mbox[0].keys():
    print(key)

X-GM-THRID
X-Gmail-Labels
Delivered-To
Received
X-Received
ARC-Seal
ARC-Message-Signature
ARC-Authentication-Results
Return-Path
Received
Received-SPF
Authentication-Results
DKIM-Signature
X-Google-DKIM-Signature
X-Gm-Message-State
MIME-Version
X-Received
Date
X-Account-Notification-Type
Feedback-ID
gmsai
X-Google-Notification-Metadata
X-Notifications
X-Notifications-Bounce-Info
Message-ID
Subject
From
To
Content-Type


## Data transformation

In [4]:
import csv

In [5]:
with open('mailbox.csv', 'w') as outputfile:
    writer = csv.writer(outputfile)
    writer.writerow(['Subject', 'From', 'Date', 'To', 'Label', 'Thread'])
    for message in mbox:
        writer.writerow([
            message['Subject'],
            message['From'],
            message['Date'],
            message['To'],
            message['X-Gmail-Labels'],
            message['X-GM-THRID']
            ]
        )

In [6]:
dfs = pd.read_csv('mailbox.csv', names=['Subject', 'From', 'Date', 'To', 'Label', 'thread'])
dfs.dtypes

Subject    str
From       str
Date       str
To         str
Label      str
thread     str
dtype: object

In [10]:
dfs['Date'] = dfs['Date'].apply(lambda x: pd.to_datetime(x, errors='coerce', utc=True))
dfs = dfs[dfs['Date'].notna() ]
dfs.to_csv('gmail.csv')

In [11]:
dfs.info()

<class 'pandas.DataFrame'>
Index: 70287 entries, 1 to 71683
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype              
---  ------   --------------  -----              
 0   Subject  70272 non-null  str                
 1   From     70287 non-null  str                
 2   Date     70287 non-null  datetime64[us, UTC]
 3   To       70251 non-null  str                
 4   Label    70284 non-null  str                
 5   thread   70287 non-null  str                
dtypes: datetime64[us, UTC](1), str(5)
memory usage: 3.8 MB


In [12]:
dfs.head(10)

,Subject,From,Date,To,Label,thread
1,Archive of Google data requested,Google Takeout <no-reply@accounts.google.com>,2026-09-17 20:55:24+00:00,dragonoids.2004@gmail.com,"Inbox,Important,Category Updates,Unread",1876613949275959936
2,Do you know the difference between you and Mag...,"""Chess.com"" <hello@chess.com>",2026-09-16 01:59:24+00:00,dragonoids.2004@gmail.com,"Inbox,Category Promotions,Unread",1876451881059910485
3,A Nashville home for you at $535K.,Redfin <listings@redfin.com>,2026-09-17 15:49:11+00:00,dragonoids.2004@gmail.com,"Inbox,Important,Category Updates,Unread",1876594683408385775
4,You're so close to hitting a 2-week streak.,Quizlet <newsletter@lifecycle.quizlet.com>,2026-09-16 13:01:34+00:00,dragonoids.2004@gmail.com,"Inbox,Category Updates,Unread",1876493542377749665
5,"Ty, get 25% off select styles when you spend $...","""New Balance Offers"" <newbalance@email.newbala...",2026-09-16 05:54:33+00:00,dragonoids.2004@gmail.com,"Inbox,Category Promotions,Unread",1876466676114685014
6,Back In Your Rotation,boohooMAN <style@e.boohooman.com>,2026-09-15 22:16:54+00:00,dragonoids.2004@gmail.com,"Inbox,Category Promotions,Unread",1876437881916376326
7,=?UTF-8?B?V2XigJl2ZQ==?= been in the lab =?UTF...,Etsy <email@email.etsy.com>,2026-09-16 15:13:25+00:00,dragonoids.2004@gmail.com,"Inbox,Category Promotions,Unread",1876501835353200092
8,"Your Request: $1,250 Scholarship!",New Scholarships | Fastweb <webmaster@fastweb....,2026-09-16 13:46:05+00:00,dragonoids.2004@gmail.com,"Inbox,Category Updates,Unread",1876496342393271275
9,Small-shop treasures await =?UTF-8?B?8J+qjg==?=,Etsy <email@email.etsy.com>,2026-09-17 18:15:13+00:00,dragonoids.2004@gmail.com,"Inbox,Important,Category Promotions,Unread",1876603872921884216
10,Your Wednesday just got way better,Pizza Hut <Promotions@my.pizzahut.com>,2026-09-16 18:20:46+00:00,dragonoids.2004@gmail.com,"Inbox,Category Promotions,Unread",1876513624600360164


## Data refactoring

In [20]:
import re
import numpy as np


def extract_email_ID(string):
    email = re.findall(r'<(.+?)>', string)
    if not email:
        email = list(filter(lambda y: '@' in y, string.split()))
    return email[0] if email else np.nan


dfs['From'] = dfs['From'].apply(lambda x: extract_email_ID(x))


myemail = 'dragonoids.2004@gmail.com'
dfs['Label'] = dfs['From'].apply(lambda x: 'Sent' if x == myemail else 'Inbox')


dfs.drop(columns='To', inplace=True)
dfs.head(10)

TypeError: expected string or bytes-like object, got 'float'